In [ ]:
pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,Dense,LSTM

In [ ]:
data = pd.read_json('News_Category_Dataset_v3.json',lines=True)

In [ ]:
data.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [ ]:
# Data Exploration

In [ ]:
data.shape

(209527, 6)

In [ ]:
data = data['short_description'].head(20000)

In [ ]:
data.isnull().sum()

np.int64(0)

In [ ]:
data.duplicated().sum()

np.int64(66)

In [ ]:
# Data Cleaning

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
# Tokenization

In [ ]:
tokenizer = Tokenizer(oov_token='<oov>')

In [ ]:
tokenizer.fit_on_texts(data)

In [ ]:
len(tokenizer.word_index)

26343

In [ ]:
data = tokenizer.texts_to_sequences(data)

In [ ]:
input_sequences =[]
for seq in data:
    if len(seq)<2:
        continue
    for i in range(1,len(seq)):
        ngram = seq[:i+1]
        input_sequences.append(ngram)

In [ ]:
max_len = max([len(i) for i in input_sequences])

In [ ]:
input  = pad_sequences(input_sequences,padding='pre',maxlen=max_len)

In [ ]:
max_len

44

In [ ]:
x = input[:,:-1]
y = input[:,-1]

In [ ]:
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
vocab_size = len(tokenizer.word_index)+1

In [ ]:
vocab_size

26344

In [ ]:
max_len = x.shape[1]

In [ ]:
max_len

43

In [ ]:
model = Sequential([Embedding(vocab_size,128,input_length=max_len),
                    LSTM(256),Dense(vocab_size,activation='softmax')])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
model.build(input_shape=(None,max_len))

In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 43, 128)        │     3,372,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 26344)          │     6,770,408 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,536,680 (40.19 MB)

 Trainable params: 10,536,680 (40.19 MB)

 Non-trainable params: 0 (0.00 B)

In [117]:
model.fit(x_train,y_train,epochs=30,validation_data=(x_test,y_test))

Epoch 1/30
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 115s 15ms/step - accuracy: 0.7810 - loss: 0.9635 - val_accuracy: 0.0943 - val_loss: 11.1575
Epoch 2/30
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 142s 15ms/step - accuracy: 0.7834 - loss: 0.9512 - val_accuracy: 0.0934 - val_loss: 11.2356
Epoch 3/30
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 115s 15ms/step - accuracy: 0.7848 - loss: 0.9407 - val_accuracy: 0.0922 - val_loss: 11.3317
Epoch 4/30
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 115s 15ms/step - accuracy: 0.7859 - loss: 0.9376 - val_accuracy: 0.0936 - val_loss: 11.3923
Epoch 5/30
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 115s 15ms/step - accuracy: 0.7865 - loss: 0.9311 - val_accuracy: 0.0917 - val_loss: 11.4873
Epoch 6/30
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 115s 15ms/step - accuracy: 0.7860 - loss: 0.9270 - val_accuracy: 0.0935 - val_loss: 11.5364
Epoch 7/30
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 115s 15ms/step - accuracy: 0.7869 - loss: 0.9251 - val_accuracy: 0.0922 - val_loss: 11.6213
Epoch 8/30
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 115s 15ms/step - acc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [118]:
import pickle

with open('tokenizer.pkl','wb') as file:
    pickle.dump(tokenizer,file)

In [119]:
model.save('nextword_model.h5')